# N04: Tree Boosting Bake-Off (Categorical Target Encoding Only)

In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
sub_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv')


In [ ]:

# 2. Base Feature Engineering
def engineer_features(df):
    df_new = df.copy()
    df_new['sleep_bin'] = pd.qcut(df_new['sleep_duration'], q=5, duplicates='drop').astype(str)
    df_new['stress_sleep_interact'] = df_new['stress_level'].astype(str) + '_' + df_new['sleep_bin']
    return df_new

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df['health_condition'])

X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

# Standard Numeric Imputation for continuous numeric features
num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 3. Model Evaluation Function (Categorical TE Only)
def evaluate_model(model_name, model_fn):
    print(f"\n{'='*50}\nEvaluating {model_name}\n{'='*50}")
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    oof_preds = np.zeros((len(train_df), 3))
    test_probs = np.zeros((len(test_df), 3))
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_raw, y_train)):
        X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[train_idx], X_train_raw[cat_cols].iloc[val_idx]
        X_tr_num, X_va_num = X_train_num_raw[train_idx], X_train_num_raw[val_idx]
        y_tr, y_va = y_train[train_idx], y_train[val_idx]
        
        fold_te_tr = []
        fold_te_val = []
        fold_te_test = []
        
        # Exact-Value Target Encoding strictly applied to categorical and categorical-interaction columns
        for col in cat_cols:
            crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
            mapping = crosstab.to_dict('index')
            mean_mapping = crosstab.mean().to_dict()
            
            tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
            fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
            
            va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
            fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
            
            te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
            fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
            
        X_tr_te = np.hstack(fold_te_tr)
        X_va_te = np.hstack(fold_te_val)
        X_te_te = np.hstack(fold_te_test)
        
        X_tr_full = np.hstack([X_tr_num, X_tr_te])
        X_va_full = np.hstack([X_va_num, X_va_te])
        X_te_full = np.hstack([X_test_num_raw, X_te_te])
        
        model = model_fn()
        sample_weights = compute_sample_weight('balanced', y_tr)
        
        if model_name == 'LightGBM':
            callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)]
            model.fit(X_tr_full, y_tr, sample_weight=sample_weights, eval_set=[(X_va_full, y_va)], callbacks=callbacks)
        elif model_name == 'CatBoost':
            model.fit(X_tr_full, y_tr, eval_set=[(X_va_full, y_va)], early_stopping_rounds=50, verbose=0)
        elif model_name == 'XGBoost':
            model.fit(X_tr_full, y_tr, sample_weight=sample_weights, eval_set=[(X_va_full, y_va)], verbose=0)
        else: # HistGradientBoosting
            model.fit(X_tr_full, y_tr, sample_weight=sample_weights)
            
        val_probs = model.predict_proba(X_va_full)
        oof_preds[val_idx] = val_probs
        
        test_probs += model.predict_proba(X_te_full) / n_splits
        
        score = balanced_accuracy_score(y_va, val_probs.argmax(axis=1))
        print(f"Fold {fold+1} Balanced Accuracy: {score:.4f}")
        scores.append(score)
        
    avg_score = np.mean(scores)
    print(f"\n>>> {model_name} Overall CV Balanced Accuracy: {avg_score:.4f} <<<")
    return avg_score, oof_preds, test_probs


In [ ]:

# 4. Bake-Off Execution
results = {}
oof_predictions = {}
test_predictions = {}

results['LightGBM'], oof_predictions['LightGBM'], test_predictions['LightGBM'] = evaluate_model(
    'LightGBM', 
    lambda: LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
)

results['CatBoost'], oof_predictions['CatBoost'], test_predictions['CatBoost'] = evaluate_model(
    'CatBoost', 
    lambda: CatBoostClassifier(iterations=1500, learning_rate=0.05, depth=6, auto_class_weights='Balanced', random_state=42, task_type='CPU')
)

results['HistGradientBoosting'], oof_predictions['HistGradientBoosting'], test_predictions['HistGradientBoosting'] = evaluate_model(
    'HistGradientBoosting', 
    lambda: HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.05, max_leaf_nodes=63, early_stopping=True, validation_fraction=0.1, n_iter_no_change=50, random_state=42)
)

results['XGBoost'], oof_predictions['XGBoost'], test_predictions['XGBoost'] = evaluate_model(
    'XGBoost', 
    lambda: XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, objective='multi:softprob', eval_metric='mlogloss', early_stopping_rounds=50)
)

print("\n" + "="*50)
print("FINAL BAKE-OFF RESULTS")
print("="*50)
for model, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model}: {score:.5f}")


In [ ]:

# 5. Blend Export
best_model_name = max(results, key=results.get)
best_test_probs = test_predictions[best_model_name]

blend_probs = np.mean([probs for probs in test_predictions.values()], axis=0)
oof_blend = np.mean([probs for probs in oof_predictions.values()], axis=0)
blend_score = balanced_accuracy_score(y_train, oof_blend.argmax(axis=1))
print(f"\nEnsemble (Uniform Blend) OOF Balanced Accuracy: {blend_score:.5f}")

sub_df['health_condition'] = le_target.inverse_transform(blend_probs.argmax(axis=1))
sub_df.to_csv('submission.csv', index=False)

sub_df_single = sub_df.copy()
sub_df_single['health_condition'] = le_target.inverse_transform(best_test_probs.argmax(axis=1))
sub_df_single.to_csv('submission_best_single_model.csv', index=False)

print(f"Saved best single model ({best_model_name}) to submission_best_single_model.csv")
print("Saved blend ensemble to submission.csv")
